# 05. Ejecución en Paralelo Multi-Caso y Comparación Masiva

Demuestra el uso de `ParallelRunner` (`ProcessPoolExecutor` con contexto `spawn`) para ejecutar en
paralelo múltiples pares `(caso, arquitectura)`, distribuir hilos/GPUs y consolidar el tablero de resultados.

In [2]:
import sys
from pathlib import Path

# Localiza PINN_V4 automáticamente aunque se ejecute desde dentro de notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Raíz PINN_V4:", ROOT)

Raíz PINN_V4: C:\usr\ths_mia_fiis\tesis_MIA_lk\tesis_MIA_2026-2_pinn


In [ ]:
from pinn_thermal.core.parallel import ParallelRunner, RunTask
from frontend.state import WorkspaceStore

store = WorkspaceStore(ROOT / "frontend" / "workspace", ROOT / "cases")
out_root = store.runs_dir / "nb05_parallel_batch"

tasks = [
    RunTask(
        name="mms_constant__M1",
        problem_source=str(ROOT / "cases" / "benchmarks" / "mms_constant" / "problem.json"),
        architecture="M1",
        output_dir=str(out_root / "mms_constant__M1"),
        config_kwargs={"adam_steps": 120, "lbfgs_steps": 10, "width": 20, "depth": 3},
        overwrite=True,
    ),
    RunTask(
        name="mms_constant__CUSPR",
        problem_source=str(ROOT / "cases" / "benchmarks" / "mms_constant" / "problem.json"),
        architecture="CUSPR",
        output_dir=str(out_root / "mms_constant__CUSPR"),
        config_kwargs={"adam_steps": 120, "lbfgs_steps": 10, "width": 20, "depth": 3},
        overwrite=True,
    ),
]

runner = ParallelRunner(max_workers=2)
summary = runner.run(tasks)
for row in summary:
    print(row)